# Databricks AutoML - Clasificación Práctica

## 🎯 Objetivo

En este notebook usaremos **Databricks AutoML** para crear un modelo de clasificación que prediga **customer churn** (cancelación de clientes).

### Lo que aprenderemos:

1. ✅ Cómo ejecutar AutoML con Python API
2. ✅ Analizar los resultados y el leaderboard
3. ✅ Comparar AutoML vs modelo manual (Decision Tree del notebook anterior)
4. ✅ Entender qué modelos probó AutoML
5. ✅ Registrar el mejor modelo en MLflow
6. ✅ Ver el notebook generado automáticamente

---

## 📊 Dataset: Customer Churn

**Problema de negocio:**  
Una empresa de telecomunicaciones quiere predecir qué clientes cancelarán su servicio.

**Features:**
* `tenure`: Meses como cliente
* `monthly_charges`: Cargo mensual ($)
* `total_charges`: Total gastado ($)
* `contract`: Tipo de contrato (Month-to-month, One year, Two year)
* `internet_service`: Tipo de internet (DSL, Fiber optic, No)
* `payment_method`: Método de pago
* `senior_citizen`: Es adulto mayor (0/1)
* Y más...

**Target:**
* `churn`: 1 = Canceló, 0 = Sigue activo

**Dataset balanceado:**
* ~73% No Churn (clase 0)
* ~27% Churn (clase 1)

---

## 🔧 Setup

Primero vamos a crear un dataset sintético de churn para este ejemplo.

In [0]:
# Crear dataset sintético de customer churn
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession

np.random.seed(42)
n_samples = 2000

# Generar features
data = {
    'customer_id': range(1, n_samples + 1),
    'tenure': np.random.randint(1, 72, n_samples),
    'monthly_charges': np.random.uniform(20, 120, n_samples),
    'total_charges': np.random.uniform(100, 8000, n_samples),
    'contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples, p=[0.5, 0.3, 0.2]),
    'internet_service': np.random.choice(['DSL', 'Fiber optic', 'No'], n_samples, p=[0.4, 0.4, 0.2]),
    'payment_method': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n_samples),
    'senior_citizen': np.random.binomial(1, 0.15, n_samples),
    'online_security': np.random.choice(['Yes', 'No', 'No internet service'], n_samples, p=[0.3, 0.5, 0.2]),
    'tech_support': np.random.choice(['Yes', 'No', 'No internet service'], n_samples, p=[0.3, 0.5, 0.2]),
}

df = pd.DataFrame(data)

# Generar target con lógica de negocio
churn_prob = 0.1
churn_prob_adjusted = np.where(df['contract'] == 'Month-to-month', 0.4, 0.1)
churn_prob_adjusted = np.where(df['tenure'] < 12, churn_prob_adjusted * 2, churn_prob_adjusted)
churn_prob_adjusted = np.where(df['monthly_charges'] > 80, churn_prob_adjusted * 1.5, churn_prob_adjusted)
churn_prob_adjusted = np.where(df['online_security'] == 'No', churn_prob_adjusted * 1.3, churn_prob_adjusted)
churn_prob_adjusted = np.clip(churn_prob_adjusted, 0, 1)
df['churn'] = np.random.binomial(1, churn_prob_adjusted)

# Convertir a Spark DataFrame
spark_df = spark.createDataFrame(df)

# Mostrar estadísticas
print("\n┌──────────────────────────────────────────────────┐")
print("│        DATASET DE CUSTOMER CHURN CREADO          │")
print("└──────────────────────────────────────────────────┘")
print(f"\n📊 Total registros: {spark_df.count():,}")
print(f"\n📋 Columnas ({len(spark_df.columns)}): {', '.join(spark_df.columns)}")

churn_dist = spark_df.groupBy('churn').count().toPandas()
print("\n🎯 Distribución de Churn:")
for _, row in churn_dist.iterrows():
    label = "No Churn (0)" if row['churn'] == 0 else "Churn (1)"
    pct = row['count'] / n_samples * 100
    print(f"  {label}: {row['count']:,} ({pct:.1f}%)")

print("\n✅ Dataset listo para AutoML")
display(spark_df.limit(5))

## 🤖 Ejecutar Databricks AutoML

### Parámetros Clave

```python
automl.classify(
    dataset=spark_df,
    target_col="churn",
    primary_metric="f1",
    timeout_minutes=15,
    max_trials=20,
    feature_store_lookups=None,
    exclude_cols=["customer_id"],
    pos_label=1
)
```

### Métricas Disponibles para Clasificación

* `f1` ⭐ (recomendado para desbalance)
* `accuracy`
* `precision`
* `recall`
* `log_loss`
* `roc_auc`

### Modelos que AutoML Probará

* Decision Tree Classifier
* Random Forest Classifier
* **XGBoost Classifier** ⭐
* **LightGBM Classifier** ⭐ (usualmente el mejor)
* Logistic Regression

---

**⚠️ Nota:** El siguiente código tomará ~10-15 minutos en ejecutar.

In [0]:
print("\n⚠️  IMPORTANTE: Databricks AutoML Python API no está disponible en Serverless Compute")
print("\n📌 Estás usando: Serverless (Spark Connect)")
print("   La API databricks.automl solo funciona en clusters clásicos\n")
print("="*80)
print("\n🔧 OPCIONES:\n")
print("1️⃣  Usar la UI de AutoML (Recomendado):")
print("    • Menú izquierdo → 'Machine Learning' → 'Experiments'")
print("    • Clic en 'Create AutoML Experiment'")
print("    • Selecciona tu dataset (guarda spark_df como tabla primero)")
print("    • Configura el experimento en la UI")
print("    • AutoML generará un notebook con el mejor modelo\n")
print("2️⃣  Cambiar a Cluster Clásico:")
print("    • Detach de serverless")
print("    • Crear/conectar a un all-purpose cluster")
print("    • Re-ejecutar este notebook en ese cluster\n")
print("3️⃣  Continuar con modelo manual:")
print("    • Saltar celdas 5-7 (que dependen de AutoML)")
print("    • Ir directo a celda 9 (modelo Random Forest manual)")
print("    • Este notebook está diseñado para funcionar sin AutoML API\n")
print("="*80)
print("\n✅ Para este notebook educativo, continuaremos con la opción 3")
print("   (modelo manual en las siguientes celdas)\n")

In [0]:
# La API de AutoML no está disponible en Serverless Compute.
# Ver celda anterior para las opciones disponibles.

print("\n⚠️  Esta celda no se puede ejecutar en Serverless Compute")
print("\n📋 Código de AutoML (requiere cluster clásico):")
print("="*80)
print("""
from databricks import automl
import mlflow

summary = automl.classify(
    dataset=spark_df,
    target_col="churn",
    primary_metric="f1",
    timeout_minutes=15,
    max_trials=20,
    exclude_cols=["customer_id"],
    pos_label=1
)
""")
print("="*80)
print("\n✅ Para continuar con este notebook educativo:")
print("   • Saltar a la celda 9 (modelo Random Forest manual)")
print("   • O usar la UI de AutoML (ver instrucciones en celda 4)\n")

## 📊 Analizar Resultados del AutoML

### 🔍 Exploración del Summary

El objeto `summary` contiene:

* `best_trial`: Información del mejor modelo
* `trials`: Lista de todos los trials ejecutados
* `experiment`: MLflow Experiment donde se guardaron los runs

### 🏆 Leaderboard de Modelos

Vamos a ver todos los modelos que AutoML probó y compararlos.

In [0]:
# Esta celda requiere el objeto 'summary' de AutoML
# Como estamos en Serverless, saltamos esta celda y continuamos con el modelo manual

print("\n⏭️  CELDA OMITIDA (requiere AutoML)")
print("\n📋 Esta celda mostraría el leaderboard de todos los modelos que AutoML probó.")
print("   En un cluster clásico, verías una tabla comparando:")
print("   • Decision Tree, Random Forest, XGBoost, LightGBM, Logistic Regression")
print("   • Con métricas: F1, Accuracy, Precision, Recall, ROC AUC\n")
print("="*80)
print("\n✅ CONTINUANDO CON MODELO MANUAL:")
print("   👉 Ir a la celda 9 para ver el Random Forest manual")
print("   👉 Celdas 10 y 12 también serán omitidas (dependen de AutoML)\n")

## 🔬 Comparación: AutoML vs Modelo Manual

### Recordatorio: Notebooks Anteriores

En los notebooks de **Aprendizaje Supervisado/Clasificación**, creamos manualmente:

* Decision Tree Classifier
* Random Forest Classifier

Con preprocesamiento manual, feature engineering, y tuning de hiperparámetros.

### Simulación de Comparación

Vamos a simular un modelo manual simple para comparar con AutoML.

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, roc_auc_score
import time

print("┌──────────────────────────────────────────────────┐")
print("│      ENTRENAR MODELO MANUAL (RANDOM FOREST)      │")
print("└──────────────────────────────────────────────────┘")

start_time = time.time()
df_manual = spark_df.toPandas()

print("\n🧹 Preprocesamiento manual...")
X = df_manual.drop(['churn', 'customer_id'], axis=1)
y = df_manual['churn']

categorical_cols = ['contract', 'internet_service', 'payment_method', 'online_security', 'tech_support']
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
print("  ✅ Categorical encoding completado")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"  ✅ Train/test split: {len(X_train)} train, {len(X_test)} test")

print("\n🌳 Entrenando Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
print("  ✅ Modelo entrenado")

print("\n📊 Evaluando en test set...")
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

manual_f1 = f1_score(y_test, y_pred)
manual_accuracy = accuracy_score(y_test, y_pred)
manual_precision = precision_score(y_test, y_pred)
manual_recall = recall_score(y_test, y_pred)
manual_roc_auc = roc_auc_score(y_test, y_pred_proba)
manual_time = time.time() - start_time

print("\n✅ Evaluación completada")
print("\n┌──────────────────────────────────────────────────┐")
print("│            RESULTADOS DEL MODELO MANUAL           │")
print("└──────────────────────────────────────────────────┘")
print(f"\nF1 SCORE: {manual_f1:.4f}")
print(f"ACCURACY: {manual_accuracy:.4f}")
print(f"PRECISION: {manual_precision:.4f}")
print(f"RECALL: {manual_recall:.4f}")
print(f"ROC AUC: {manual_roc_auc:.4f}")
print(f"\n⏱️  TIEMPO: {manual_time:.1f} segundos")

In [0]:
# Esta celda requiere el objeto 'summary' de AutoML
# Como estamos en Serverless, saltamos esta celda y continuamos con el modelo manual

print("\n⏭️  CELDA OMITIDA (requiere AutoML)")
print("\n📋 Esta celda compararía las métricas de AutoML vs el modelo manual.")
print("   En un cluster clásico, verías:")
print("   • Tabla comparativa de F1, Accuracy, Precision, Recall, ROC AUC")
print("   • Qué modelo ganó y por cuánto")
print("   • Comparación de tiempo de ejecución")
print("   • Comparación de esfuerzo (líneas de código)\n")
print("="*80)
print("\n✅ RESULTADOS DEL MODELO MANUAL (de la celda 9):")
print(f"   F1 Score: {manual_f1:.4f}")
print(f"   Accuracy: {manual_accuracy:.4f}")
print(f"   Precision: {manual_precision:.4f}")
print(f"   Recall: {manual_recall:.4f}")
print(f"   ROC AUC: {manual_roc_auc:.4f}")
print(f"   Tiempo: {manual_time:.1f} segundos\n")

## 📚 Explorar el Notebook Generado por AutoML

### 🔍 Lo que contiene el notebook:

1. **Data Exploration**:
   - Estadísticas descriptivas
   - Visualizaciones de distribuciones
   - Correlaciones
   - Detección de outliers

2. **Preprocesamiento Automático**:
   - Imputación de valores faltantes
   - Encoding de categóricas (One-Hot, Label, Target)
   - Scaling de numéricas
   - Feature engineering

3. **Model Training**:
   - Código completo del mejor modelo
   - Hiperparámetros optimizados
   - Train/validation split

4. **Evaluation**:
   - Métricas detalladas
   - Matriz de confusión
   - Feature importance
   - ROC Curve, Precision-Recall Curve

5. **SHAP Explainability**:
   - Feature contributions
   - Interpretabilidad del modelo

### ✅ El notebook es 100% reproducible y editable

**Puedes:**
* Copiar el código
* Modificar hiperparámetros
* Agregar features custom
* Reentrenar con datos nuevos

Veamos la ruta del notebook:

In [0]:
# Esta celda requiere el objeto 'summary' de AutoML
# Como estamos en Serverless, saltamos esta celda y continuamos con el modelo manual

print("\n⏭️  CELDA OMITIDA (requiere AutoML)")
print("\n📋 Esta celda mostraría información del notebook generado automáticamente por AutoML.")
print("   En un cluster clásico, verías:")
print("   • Ruta del notebook generado")
print("   • MLflow Run ID del mejor modelo")
print("   • Model URI para carga")
print("   • Instrucciones para explorar el código completo\n")
print("="*80)
print("\n📚 CONTENIDO QUE AUTOML GENERA EN SU NOTEBOOK:")
print("   • Data exploration completo")
print("   • Preprocesamiento automático")
print("   • Código del modelo ganador")
print("   • Hiperparámetros optimizados")
print("   • Evaluación con múltiples métricas")
print("   • Feature importance")
print("   • SHAP explainability")
print("   • 100% reproducible y editable\n")
print("="*80)
print("\n✅ Para experimentar con AutoML completo:")
print("   • Opción 1: Usar la UI de AutoML (ver celda 4)")
print("   • Opción 2: Cambiar a un cluster clásico y re-ejecutar este notebook\n")

## 📝 Conclusiones del Notebook

### ✅ Lo que aprendimos:

1. **AutoML es increíblemente eficiente**:
   - 5 líneas de código
   - Prueba múltiples algoritmos
   - Optimización automática de hiperparámetros
   - Genera notebook explicativo

2. **Performance comparable o superior** a modelos manuales:
   - Especialmente para datasets tabulares
   - LightGBM y XGBoost suelen ser los ganadores

3. **Integración perfecta con MLflow**:
   - Todos los runs trackeados automáticamente
   - Fácil comparación de modelos
   - Model Registry integrado

4. **Excelente punto de partida**:
   - Usa AutoML para baseline
   - Analiza el notebook generado
   - Itera manualmente desde ahí si es necesario

---

### 🚀 Próximos Pasos

**En el siguiente notebook** (`Databricks_AutoML_Regresion.ipynb`):
* AutoML para regresión
* Predicción de precios
* Métricas de regresión (RMSE, MAE, R²)

**Luego**:
* Genie Code como asistente de ML
* MLflow tracking end-to-end
* Feature Store

---

### 💡 Key Takeaways

> **"AutoML no reemplaza a los Data Scientists, sino que los hace 10x más productivos."**

✅ **Cuándo usar AutoML:**
* Baseline rápido
* Exploración inicial
* Proyectos con poco tiempo
* Democratización de ML

🔧 **Cuándo iterar manualmente:**
* Necesitas features custom
* Arquitecturas especializadas
* Performance crítico
* Deep understanding del problema

**Mejor práctica:** 🤝 **AutoML + Manual = Winning Combination**